In [1]:
from sqlalchemy import create_engine
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from dotenv import load_dotenv
import datetime
import os
import mplfinance as mpf
import re
import utils

IndentationError: unexpected indent (utils.py, line 10)

In [ ]:
load_dotenv()
# f"postgres+psycopg2://{[user]}:{[pass]}@{[host}:{[port]}/{[dbname]}"
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER')}:{os.getenv('POSTGRES_PASSWORD')}"
    f"@{os.getenv('POSTGRES_HOST')}:{os.getenv('POSTGRES_PORT')}/{os.getenv('POSTGRES_DB')}"
)

In [ ]:
start_time='08:30:00'
end_time='09:30:00'

In [ ]:
query_min = "SELECT * FROM gold.active_1min WHERE bar_time BETWEEN %(start)s AND %(end)s ORDER BY bar_date, bar_time"
df_min = pd.read_sql(query_min,engine, params={"start": start_time, "end": end_time})

Volume Analysis

In [ ]:
df_min = utils.add_winsorized_col(df_min,'volume')
volume_agg = utils.groupby(df_min,'bar_time','volume_winsorized')
xlabel_bins = 16
hist_bins = 30

utils.get_basic_info(df_min,'volume_winsorized')
utils.frequency_hist(df_min,'volume_winsorized',hist_bins,'Volume')

utils.bar_chart(volume_agg,'mean','1 Minute Interval', 'Average Volume',tickbin=xlabel_bins)
utils.bar_chart(volume_agg,'std','1 Minute Interval', 'STD of Volume',tickbin=xlabel_bins)

Swing Analysis

In [ ]:
df_swings= utils.get_swings(df_min)
df_swings['profit'] = np.where(
    df_swings['trend']=='UP', 
    abs(df_swings['open']-df_swings['high']), 
    abs(df_swings['open']-df_swings['low'])
)
df_swings = utils.add_winsorized_col(df_swings,'profit')

In [ ]:
def analyze_swings_by_time(df,bucket,timeframe,target):
    df_agg = utils.groupby(df,'bucket_'+bucket+'min',target)
    utils.bar_chart(df_agg,'count',timeframe+' Minute Interval - '+bucket+' Minute Time Frame','Number of Swings')
    utils.bar_chart(df_agg,'mean',timeframe+' Minute Interval - '+bucket+' Minute Time Frame','Average Swing Magnitude')
    utils.bar_chart(df_agg,'std',timeframe+' Minute Interval - '+bucket+' Minute Time Frame','STD of Swing Magnitude')

In [ ]:
analyze_swings_by_time(df_swings,'5','1','profit_winsorized')

Analyze Swings

In [ ]:
gb_profit = utils.groupby(df_swings,'profit','profit')
gb_profit_win = utils.groupby(df_swings,'profit_winsorized','profit_winsorized')

In [ ]:
utils.bar_chart(gb_profit,'count','Potential Profit Magnitude','Frequency',angle=0,tickbin=10)
utils.bar_chart(gb_profit_win,'count','Potential Profit Magnitude','Frequency',angle=0,tickbin=10)

In [ ]:
utils.get_basic_info(df_swings,'profit_winsorized')